# YOLOv9-S a 640 px — ThermoSolar-PV en Kaggle (T4)

**Objetivo:** ejecutar el **P0** del plan: reentrenar YOLOv9-S a **640×640** con el split actual, `batch=16`, AMP, `patience=50`, `cos_lr=True`, `close_mosaic=10`, `save_period=10` y checkpoint/resume para sobrevivir a las sesiones de Kaggle.

Este notebook está pensado para **Kaggle Notebooks**, acelerador **GPU T4 ×2** (16 GB por GPU) e Internet activado. Por defecto entrena en **una sola T4** (`DEVICE="0"`), que es la opción más estable en notebooks. La segunda GPU se puede usar con DDP, pero no es el valor por defecto.

---

## Dos órdenes de clases: el del dataset es el correcto

El `best.pt` antiguo (y lo que antes tenían `solar_data.yaml` / `main.py`) usa este orden, que resultó estar **permutado**:

| id | clase |
|---:|---|
| 0 | Single Hotspot |
| 1 | Multi Hotspots |
| 2 | Single Diode |
| 3 | Multi Diode |
| 4 | Single Bypassed Substring |
| 5 | Multi Bypassed Substring |
| 6 | String Open Circuit |
| 7 | String Reversed Polarity |

El `data.yaml` del dataset usa **otro orden** (orden alfabético de nombres), y ese es el correcto. Se confirmó con `labels.jpg`: `Multi Hotspots` (5081) y `Single Hotspot` (7311) son las clases dominantes, y las cajas etiquetadas marcan un punto caliente como `Single Hotspot` y varios como `Multi Hotspots`. Con el orden antiguo, “Single Hotspot” habría tenido solo 357 instancias y “Multi Bypassed Substring” 7311, algo que no cuadra con las imágenes.

**El error estaba en los nombres, no en los IDs.** `main.py`, `solar_data.yaml` y `src/App.tsx` ya se actualizaron al orden del dataset.

Política por defecto (`CLASS_ORDER_POLICY="dataset"`):

- El notebook lee el `data.yaml`, comprueba que las 8 clases son las mismas y **adopta su orden**.
- Genera un `data.yaml` resuelto con ese orden y lo usa para entrenar.
- **Descarta el `best.pt` antiguo**: no lo usa para baseline ni para inicializar; entrena desde `yolov9s.pt`.
- Si el orden coincide, puedes poner `RUN_BASELINE=True` para el baseline 320 vs 640.

> Si entrenas con el orden del dataset, actualiza `CLASS_NAMES` en `main.py` (y `solar_data.yaml`) antes de desplegar. No mezcles el `best.pt` viejo con el nuevo.

---

## Qué hace el notebook

1. Localiza el dataset en `/kaggle/input` (soporta `ImageSet/train/images` y `images/train`).
2. Verifica `data.yaml`, nombres y estructura; genera un `data.yaml` resuelto.
3. Cuenta instancias por clase e imágenes por clase; revisa labels corruptos, cajas degeneradas, imágenes que no son 640×640 y co-ocurrencia Single/Multi.
4. Audita **duplicados exactos y near-duplicates** entre train/val/test con pHash (el dataset contiene muestras aumentadas; si los aumentos se repartieron antes del split, hay fuga de información).
5. Opcionalmente reconstruye un split **group-safe** por grupos de near-duplicates.
6. Opcionalmente evalúa el `best.pt` antiguo a **320** y **640**, solo si su orden de clases coincide con el `data.yaml`.
7. Entrena YOLOv9-S a 640 con checkpoint/resume.
8. Evalúa el `best.pt` final, exporta métricas por clase, AP50/AP75/AP50-95, matriz de confusión **con etiquetas** y comparación contra el baseline.
9. Guarda `best.pt`, `last.pt`, reportes y un ZIP descargable.

---

## Antes de ejecutar

- Sube el dataset a Kaggle (idealmente como **Kaggle Dataset privado**). El notebook espera la estructura del repo: `ImageSet/train/images`, `ImageSet/train/labels`, `ImageSet/valid/...`, etc. También acepta `train/images` directamente.
- El `best.pt` antiguo es opcional. Solo úsalo si el orden del `data.yaml` coincide con el suyo; si no, el notebook lo descarta y entrena desde `yolov9s.pt`.
- En **Settings**: Accelerator = `GPU T4 x2`, Internet = `On`.
- Si la sesión se corta, vuelve a abrir y ejecuta de nuevo: con `AUTO_RESUME=True` retomará desde `last.pt` si `/kaggle/working` conservó el run.
- Tiempo esperado: 200 epochs a 640 en una T4 pueden ocupar 8–15 h. `TIME_LIMIT_H=8.5` detiene limpiamente antes del límite de sesión y deja `last.pt` para reanudar.

> **Licencias:** Ultralytics es AGPL-3.0. El dataset de Kaggle declara en sus metadatos **CC BY-NC 4.0**, aunque su descripción dice MIT. Antes de un uso comercial, aclara ambas cosas. No uses Soup para esto.


## 0. Entorno

Fijamos `ultralytics==8.4.46`, la versión del run actual. Si Kaggle ya la tiene, la instalación es rápida. No actualizamos `torch`: usamos el que trae la imagen de Kaggle con CUDA.


In [ ]:
%pip install -q --no-input "ultralytics==8.4.46"

## 1. Configuración única

Cambia solo esta celda para cada experimento. Los valores por defecto son el **run A (P0: 640, sin class weighting)**.


In [ ]:
# =============================================================================
# CONFIGURACIÓN DEL EXPERIMENTO
# =============================================================================
from pathlib import Path
import os
import re
import json
import random
import shutil
import subprocess
import sys
import time
import hashlib
import zipfile
import warnings
from collections import Counter, defaultdict
from datetime import datetime

SEED = 0

# --- Entrenamiento -----------------------------------------------------------
IMG_SIZE = 640
EPOCHS = 200
BATCH = 16                  # 16 en 1×T4; si DEVICE="0,1", usa 32 (16 por GPU)
PATIENCE = 50
WORKERS = 4
DEVICE = "0"                # "0" = 1×T4. "0,1" = DDP en T4×2 (más frágil en notebooks)
TIME_LIMIT_H = 8.5          # 0 = sin tope. 8.5 h deja margen para la sesión de Kaggle
SAVE_PERIOD = 10
CLOSE_MOSAIC = 10
CLS_PW = 0.0                # 0.0 = baseline; 0.25-0.5 = experimento B para clases raras
LR0 = 0.01
RUN_NAME = "yolov9s_640_thermal"
INIT_WEIGHTS = "yolov9s.pt"  # "best.pt" solo si el orden coincide con el data.yaml
AUTO_RESUME = True
RESUME_WEIGHTS = None       # ruta a last.pt si reabres sesión y no está en /kaggle/working
RESUME_TOTAL_EPOCHS = EPOCHS  # schedule total al reanudar un run cortado por TIME_LIMIT_H
RESUME_KEEP_MOSAIC_CLOSED = True  # el run ya cerró mosaic; no lo reabras al reanudar
BASELINE_WEIGHTS = None     # best.pt del run 320; solo si el orden coincide

# --- Evaluación / análisis ---------------------------------------------------
RUN_BASELINE = False        # True solo si el data.yaml coincide con el orden del best.pt antiguo
RUN_TEST_EVAL = True        # evalúa el split test si tiene labels
RUN_TTA_EVAL = False        # TTA es 3× más lento; mide primero sin TTA
REBUILD_GROUP_SAFE_SPLIT = False  # True si la fuga por aumentos es alta
LEAK_HAMMING = 6            # 0 = solo duplicados exactos; 6 = near-duplicates
LEAK_EXAMPLES = True
EXPORT_ONNX = False

# --- Rutas -------------------------------------------------------------------
DATASET_ROOT_OVERRIDE = None  # p.ej. Path("/kaggle/input/solarpanel/ImageSet")
DATASET_SLUG = "pkdarabi/solarpanel"
WORK = Path("/kaggle/working")
PROJECT = WORK / "runs"
REPORTS = WORK / "reports"
ARTIFACTS = WORK / "artifacts"
BACKUP_DIR = WORK / "checkpoints"

# --- Orden del best.pt ANTIGUO (solo referencia) -----------------------------
# El orden activo se toma del data.yaml del dataset en la sección 3.
CHECKPOINT_NAMES = [
    "Single Hotspot",
    "Multi Hotspots",
    "Single Diode",
    "Multi Diode",
    "Single Bypassed Substring",
    "Multi Bypassed Substring",
    "String Open Circuit",
    "String Reversed Polarity",
]
EXPECTED_NAMES = CHECKPOINT_NAMES.copy()

# "dataset": adopta el orden del data.yaml y entrena desde yolov9s.pt.
# "checkpoint": aborta si el data.yaml no coincide con el orden del best.pt antiguo.
CLASS_ORDER_POLICY = "dataset"
if CLASS_ORDER_POLICY not in {"dataset", "checkpoint"}:
    raise ValueError('CLASS_ORDER_POLICY debe ser "dataset" o "checkpoint".')

# --- Aumentos para térmicas ------------------------------------------------
# hsv_h/hsv_s = 0 porque el dataset es escala de grises/colormap térmico.
AUG = dict(
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.3,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
)

DEVICE_LIST = [int(x) for x in str(DEVICE).split(",") if x.strip() != ""]
if not DEVICE_LIST:
    raise ValueError('DEVICE está vacío. Usa DEVICE="0" (una GPU) o DEVICE="0,1" (DDP).')
if len(DEVICE_LIST) > 1 and BATCH < 2 * len(DEVICE_LIST):
    warnings.warn("Con DDP el batch es total; usa BATCH=32 para 16 por GPU.")

for p in (PROJECT, REPORTS, ARTIFACTS, BACKUP_DIR):
    p.mkdir(parents=True, exist_ok=True)

print(f"Timestamp: {datetime.now().isoformat(timespec='seconds')}")
print(f"Run: {RUN_NAME} | imgsz={IMG_SIZE} | epochs={EPOCHS} | batch={BATCH} | device={DEVICE_LIST}")
print(f"Time limit: {TIME_LIMIT_H} h | patience={PATIENCE} | cls_pw={CLS_PW} | lr0={LR0}")
print(f"Dataset esperado: {DATASET_SLUG}")


## 2. Verificación de GPU y versiones


In [ ]:
import math
import torch
import ultralytics
import cv2
import numpy as np
import pandas as pd
import yaml
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display, FileLink

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("ultralytics:", ultralytics.__version__)
print("opencv:", cv2.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

assert torch.cuda.is_available(), (
    "No hay GPU. En Kaggle: Settings → Accelerator → GPU T4 x2, y vuelve a ejecutar."
)
print("GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {props.name} | {props.total_memory / 1024**3:.1f} GB")

if len(DEVICE_LIST) > 1:
    assert torch.cuda.device_count() >= len(DEVICE_LIST), "DEVICE pide más GPUs de las disponibles."

if ultralytics.__version__ != "8.4.46":
    warnings.warn(
        f"Se esperaba ultralytics==8.4.46 y hay {ultralytics.__version__}. "
        "Revisa que cls_pw/time sigan existiendo antes de confiar en la reproducibilidad."
    )

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

# --- Utilidades compartidas ---------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")


def find_attached_pt(preferred=None):
    """Devuelve el .pt indicado o el mejor candidato dentro de /kaggle/input."""
    if preferred:
        p = Path(preferred)
        if p.exists():
            return p
        raise FileNotFoundError(f"No existe BASELINE_WEIGHTS: {preferred}")
    if not INPUT_ROOT.exists():
        return None
    cands = [p for p in INPUT_ROOT.rglob("*.pt") if p.is_file()]
    if not cands:
        return None
    cands.sort(key=lambda p: (p.name.lower() != "best.pt", -p.stat().st_size))
    print("Checkpoints encontrados en /kaggle/input:")
    for p in cands:
        print(f"  {p} ({p.stat().st_size / 1024**2:.1f} MB)")
    return cands[0]


def per_class_table(metrics):
    """DataFrame por clase a partir de un DetMetrics de Ultralytics."""
    rows = []
    names = metrics.names
    for i, c in enumerate(metrics.box.ap_class_index):
        rows.append({
            "class_id": int(c),
            "class_name": names[int(c)],
            "precision": float(metrics.box.p[i]),
            "recall": float(metrics.box.r[i]),
            "mAP50": float(metrics.box.ap50[i]),
            "mAP75": float(metrics.box.all_ap[i, 5]),
            "mAP50-95": float(metrics.box.ap[i]),
        })
    present = {r["class_id"] for r in rows}
    for c in range(len(names)):
        if c not in present:
            rows.append({
                "class_id": c,
                "class_name": names[c],
                "precision": np.nan,
                "recall": np.nan,
                "mAP50": np.nan,
                "mAP75": np.nan,
                "mAP50-95": np.nan,
            })
    return pd.DataFrame(rows).sort_values("class_id").reset_index(drop=True)

def json_safe(obj):
    """Convierte tipos de numpy y NaN/Inf a algo que json.dump escriba como JSON válido."""
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.generic):
        obj = obj.item()
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if isinstance(obj, np.ndarray):
        return json_safe(obj.tolist())
    return obj


## 3. Localización y verificación del dataset

El dataset de Kaggle suele montarse como `/kaggle/input/<slug>/...`. Este bloque busca automáticamente una raíz con `train` y `valid` (o `val`), tanto en formato `train/images` como `images/train`. Si el dataset viene en ZIP, lo descomprime en `/kaggle/working`.

Si el `data.yaml` usa un orden de clases distinto al del `best.pt` antiguo, el notebook **adopta el orden del dataset** (los labels mandan), desactiva el baseline del `best.pt` antiguo y entrena desde `yolov9s.pt`.


In [ ]:
INPUT_ROOT = Path("/kaggle/input")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def find_layout(root: Path):
    """Devuelve {split: (images_dir, labels_dir)} para layouts YOLO comunes."""
    root = Path(root)
    layout = {}
    aliases = {"train": ["train"], "val": ["valid", "val"], "test": ["test"]}
    for key, names in aliases.items():
        for name in names:
            candidates = [
                (root / name / "images", root / name / "labels"),
                (root / "images" / name, root / "labels" / name),
            ]
            for img_dir, lbl_dir in candidates:
                if img_dir.is_dir() and lbl_dir.is_dir():
                    layout[key] = (img_dir, lbl_dir)
                    break
            if key in layout:
                break
    return layout if "train" in layout and "val" in layout else None


def normalize_root(root: Path):
    root = Path(root)
    if find_layout(root):
        return root
    if root.is_dir():
        for child in sorted(root.iterdir()):
            if child.is_dir() and find_layout(child):
                return child
    return None


def count_images(d: Path) -> int:
    return sum(1 for p in Path(d).iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS)


def find_candidates(base: Path):
    out = []
    if not Path(base).exists():
        return out
    for p in Path(base).rglob("*"):
        if p.is_dir():
            layout = find_layout(p)
            if layout:
                out.append((p, layout))
    # dedupe
    seen = set()
    unique = []
    for p, layout in out:
        key = str(p.resolve())
        if key not in seen:
            seen.add(key)
            unique.append((p, layout))
    return unique


def choose_candidate(candidates):
    if not candidates:
        return None
    scored = []
    for p, layout in candidates:
        n_train = count_images(layout["train"][0])
        has_yaml = (p / "data.yaml").exists() or (p.parent / "data.yaml").exists()
        score = n_train + (100_000 if has_yaml else 0)
        scored.append((score, p, layout))
    scored.sort(key=lambda x: x[0], reverse=True)
    if len(scored) > 1:
        print("Candidatos encontrados:")
        for score, p, layout in scored:
            print(f"  {p} | train={count_images(layout['train'][0])} | yaml={((p/'data.yaml').exists())}")
        print(f"Elegido: {scored[0][1]}")
    return scored[0][1], scored[0][2]


def try_unzip():
    if not INPUT_ROOT.exists():
        return None
    zips = sorted(INPUT_ROOT.rglob("*.zip"), key=lambda p: p.stat().st_size, reverse=True)
    if not zips:
        return None
    dest = WORK / "dataset_unzipped"
    dest.mkdir(parents=True, exist_ok=True)
    z = zips[0]
    print(f"Descomprimiendo {z} ({z.stat().st_size / 1024**2:.0f} MB) en {dest} ...")
    with zipfile.ZipFile(z) as zf:
        zf.extractall(dest)
    return dest


# 1) Override explícito
chosen = None
if DATASET_ROOT_OVERRIDE:
    override = Path(DATASET_ROOT_OVERRIDE)
    root = normalize_root(override)
    if root is not None:
        chosen = (root, find_layout(root))
    else:
        # Acepta también un override a una carpeta padre (busca debajo de ella).
        chosen = choose_candidate(find_candidates(override))
        if chosen is None:
            raise FileNotFoundError(
                f"DATASET_ROOT_OVERRIDE no contiene train/valid (ni directamente ni en subcarpetas): {override}"
            )
else:
    # 2) Búsqueda en /kaggle/input
    chosen = choose_candidate(find_candidates(INPUT_ROOT))

    # 3) Fallback: descomprimir ZIP y buscar dentro
    if chosen is None:
        unzipped = try_unzip()
        if unzipped is not None:
            chosen = choose_candidate(find_candidates(unzipped))

if chosen is None:
    raise FileNotFoundError(
        "No se encontró el dataset. Añade el Kaggle Dataset al notebook o define "
        "DATASET_ROOT_OVERRIDE con la carpeta que contiene train/ y valid/."
    )

DATASET_ROOT, SPLITS = chosen
DATASET_ROOT = Path(DATASET_ROOT)
SPLITS = {k: (Path(v[0]), Path(v[1])) for k, v in SPLITS.items()}

# Test sin labels: se omite y se avisa.
if (DATASET_ROOT / "test" / "images").is_dir() and "test" not in SPLITS:
    print("Aviso: hay test/images pero no test/labels; se omite la evaluación test.")


def find_data_yaml(root: Path):
    root = Path(root)
    candidates = [
        root / "data.yaml",
        root / "data.yml",
        root.parent / "data.yaml",
        root.parent / "data.yml",
    ]
    for p in candidates:
        if p.exists():
            return p
    for p in sorted(root.glob("*.yaml")):
        if "data" in p.name.lower():
            return p
    return None


def read_names(path: Path):
    if path is None or not path.exists():
        return None
    d = yaml.safe_load(path.read_text(encoding="utf-8"))
    names = d.get("names") if isinstance(d, dict) else None
    if isinstance(names, dict):
        try:
            return [names[k] for k in sorted(names, key=lambda x: int(x))]
        except Exception:
            return [names[k] for k in names]
    if isinstance(names, list):
        return names
    return None


def canon_name(s: str) -> str:
    """Normaliza camelCase, espacios, paréntesis, plurales y sinónimos.

    Ejemplos:
      "Multi Bypassed Substring" -> "multibypass"
      "MultiByPassed"            -> "multibypass"
      "String (Open Circuit)"    -> "stringopencircuit"
      "StringOpenCircuit"        -> "stringopencircuit"
    """
    s = str(s).lower()
    s = s.replace("bypassed", "bypass")
    s = s.replace("substring", "")
    s = s.replace("hotspots", "hotspot")
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s


ORIGINAL_YAML = find_data_yaml(DATASET_ROOT)
names_found = read_names(ORIGINAL_YAML)

if names_found is None:
    warnings.warn(
        "No se encontró data.yaml con names. Se usará el orden del best.pt antiguo; "
        "verifica que los labels usen ese mismo orden."
    )
    names_found = [str(x) for x in CHECKPOINT_NAMES]
    ORDER_SOURCE = "checkpoint (sin data.yaml)"
else:
    names_found = [str(x) for x in names_found]
    ORDER_SOURCE = str(ORIGINAL_YAML)

found_norm = [canon_name(x) for x in names_found]
expected_norm = [canon_name(x) for x in CHECKPOINT_NAMES]

unknown = sorted(set(found_norm) - set(expected_norm))
missing = sorted(set(expected_norm) - set(found_norm))
if len(found_norm) != len(expected_norm) or unknown or missing:
    print("\nClases encontradas en data.yaml:")
    for i, n in enumerate(names_found):
        print(f"  {i}: {n}")
    print("\nClases esperadas (best.pt antiguo):")
    for i, n in enumerate(CHECKPOINT_NAMES):
        print(f"  {i}: {n}")
    raise ValueError(
        "El data.yaml no contiene exactamente las mismas 8 clases. "
        f"Desconocidas: {unknown}. Faltantes: {missing}. "
        "Revisa el data.yaml o ajusta CHECKPOINT_NAMES."
    )

ORDER_MATCHES_CHECKPOINT = found_norm == expected_norm
OLD_BEST_COMPATIBLE = ORDER_MATCHES_CHECKPOINT

if not ORDER_MATCHES_CHECKPOINT:
    print("\n" + "=" * 78)
    print("El data.yaml usa un ORDEN DE CLASES distinto al best.pt antiguo.")
    print("Se respeta el DATASET: los labels están generados con este orden.")
    print("El best.pt antiguo NO se usará para baseline ni para inicializar.")
    print("=" * 78)
    print("\nOrden del DATASET (activo):")
    for i, n in enumerate(names_found):
        print(f"  {i}: {n}")
    print("\nOrden del best.pt ANTIGUO (solo referencia):")
    for i, n in enumerate(CHECKPOINT_NAMES):
        print(f"  {i}: {n}")

    if CLASS_ORDER_POLICY == "checkpoint":
        raise ValueError(
            "CLASS_ORDER_POLICY='checkpoint' y el data.yaml usa otro orden. "
            "Cambia a 'dataset' para respetar los labels, o corrige el data.yaml."
        )

    # Política "dataset": el data.yaml y sus labels mandan.
    RUN_BASELINE = False
    BASELINE_WEIGHTS = None
    if str(INIT_WEIGHTS) == "best.pt":
        INIT_WEIGHTS = "yolov9s.pt"
        print("\nINIT_WEIGHTS cambiado a 'yolov9s.pt' para entrenar desde cero.")
    warnings.warn(
        "Se adopta el orden del data.yaml. Baseline del best.pt antiguo desactivado. "
        "Actualiza CLASS_NAMES en main.py antes de desplegar el nuevo best.pt."
    )

# Nombres legibles, en el orden real del dataset. Los IDs de los labels no se tocan.
PRETTY_BY_CANON = {canon_name(n): n for n in CHECKPOINT_NAMES}
EXPECTED_NAMES = [
    PRETTY_BY_CANON.get(c, raw) for raw, c in zip(names_found, found_norm)
]
CLASS_NAMES = list(EXPECTED_NAMES)

print("\nOrden de clases activo:")
for i, n in enumerate(EXPECTED_NAMES):
    print(f"  {i}: {n}")
print("Fuente del orden:", ORDER_SOURCE)
print("Compatible con el best.pt antiguo:", OLD_BEST_COMPATIBLE)
print("Política:", CLASS_ORDER_POLICY)

with open(REPORTS / "class_order.json", "w", encoding="utf-8") as f:
    json.dump({
        "policy": CLASS_ORDER_POLICY,
        "source": ORDER_SOURCE,
        "matches_checkpoint": bool(ORDER_MATCHES_CHECKPOINT),
        "active_names": EXPECTED_NAMES,
        "checkpoint_names": CHECKPOINT_NAMES,
    }, f, indent=2, ensure_ascii=False)

# data.yaml resuelto, con nombres canónicos y rutas del layout encontrado
def rel_or_abs(p: Path, root: Path) -> str:
    try:
        return str(Path(p).relative_to(root))
    except ValueError:
        return str(p)


resolved = {
    "path": str(DATASET_ROOT),
    "train": rel_or_abs(SPLITS["train"][0], DATASET_ROOT),
    "val": rel_or_abs(SPLITS["val"][0], DATASET_ROOT),
    "names": {i: n for i, n in enumerate(EXPECTED_NAMES)},
}
if "test" in SPLITS:
    resolved["test"] = rel_or_abs(SPLITS["test"][0], DATASET_ROOT)

DATA_YAML = WORK / "data_solar_640.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(resolved, f, sort_keys=False, allow_unicode=True)

print("\nDataset root:", DATASET_ROOT)
for split, (img_dir, lbl_dir) in SPLITS.items():
    print(f"  {split:5s}: {count_images(img_dir):5d} imágenes | labels={lbl_dir}")
print(f"\ndata.yaml resuelto: {DATA_YAML}")
print(DATA_YAML.read_text(encoding="utf-8"))


## 4. Conteo de instancias y QA de labels

Esto responde a una de las preguntas pendientes del handoff: **cuántas instancias hay por clase**. También detecta labels mal formados, cajas degeneradas, imágenes que no son 640×640 y cuántas imágenes contienen cada clase.

Además calcula la co-ocurrencia de los pares Single/Multi. Si muchas imágenes tienen a la vez `Single Hotspot` y `Multi Hotspots`, el problema no es solo de datos: es de formulación de clases.


In [ ]:
def list_images(d: Path):
    return sorted(
        p for p in Path(d).iterdir()
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )


def list_labels(d: Path):
    return sorted(
        p for p in Path(d).glob("*.txt")
        if p.name.lower() not in {"classes.txt"}
    )


def scan_split(split_key: str, img_dir: Path, lbl_dir: Path):
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    imgs = list_images(img_dir)
    lbls = list_labels(lbl_dir)
    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}
    nc = len(EXPECTED_NAMES)

    stats = {
        "images": len(imgs),
        "label_files": len(lbls),
        "missing_labels": 0,
        "orphan_labels": len(lbl_stems - img_stems),
        "empty_labels": 0,
        "boxes": 0,
        "malformed_lines": 0,
        "bad_class_ids": 0,
        "bad_coords": 0,
        "tiny_boxes": 0,
        "images_without_boxes": 0,
        "corrupt_images": 0,
        "non_640": 0,
    }
    instances = np.zeros(nc, dtype=np.int64)
    image_presence = np.zeros(nc, dtype=np.int64)
    box_areas = {c: [] for c in range(nc)}
    class_sets = []
    per_image_boxes = []
    sizes = Counter()
    samples = {k: [] for k in ("missing", "orphan", "malformed", "bad_class", "bad_coord", "tiny", "corrupt")}

    for n, img in enumerate(imgs, 1):
        w = h = None
        try:
            with Image.open(img) as im:
                w, h = im.size
            sizes[(w, h)] += 1
            if (w, h) != (IMG_SIZE, IMG_SIZE):
                stats["non_640"] += 1
        except Exception as e:
            stats["corrupt_images"] += 1
            if len(samples["corrupt"]) < 5:
                samples["corrupt"].append((str(img), str(e)))

        lbl = lbl_dir / f"{img.stem}.txt"
        if not lbl.exists():
            stats["missing_labels"] += 1
            if len(samples["missing"]) < 5:
                samples["missing"].append(str(img))
            per_image_boxes.append(0)
            class_sets.append(set())
            continue

        text = lbl.read_text(encoding="utf-8", errors="replace")
        lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
        if not lines:
            stats["empty_labels"] += 1
            per_image_boxes.append(0)
            class_sets.append(set())
            continue

        classes_here = set()
        n_valid = 0
        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                stats["malformed_lines"] += 1
                if len(samples["malformed"]) < 5:
                    samples["malformed"].append((str(lbl), line[:120]))
                continue
            try:
                cls_f = float(parts[0])
                cls = int(cls_f)
                if abs(cls_f - cls) > 1e-6:
                    raise ValueError
                vals = [float(x) for x in parts[1:]]
            except Exception:
                stats["malformed_lines"] += 1
                if len(samples["malformed"]) < 5:
                    samples["malformed"].append((str(lbl), line[:120]))
                continue

            if cls < 0 or cls >= nc:
                stats["bad_class_ids"] += 1
                if len(samples["bad_class"]) < 5:
                    samples["bad_class"].append((str(lbl), cls))
                continue

            x, y, bw, bh = vals
            if (not all(np.isfinite(v) for v in vals)) or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                stats["bad_coords"] += 1
                if len(samples["bad_coord"]) < 5:
                    samples["bad_coord"].append((str(lbl), line[:120]))
                continue

            if bw < 2 / IMG_SIZE or bh < 2 / IMG_SIZE:
                stats["tiny_boxes"] += 1
                if len(samples["tiny"]) < 5:
                    samples["tiny"].append((str(lbl), line[:120]))

            instances[cls] += 1
            classes_here.add(cls)
            n_valid += 1
            if w and h:
                box_areas[cls].append(bw * w * bh * h)

        for c in classes_here:
            image_presence[c] += 1
        stats["boxes"] += n_valid
        per_image_boxes.append(n_valid)
        class_sets.append(classes_here)
        if n_valid == 0:
            stats["images_without_boxes"] += 1
        if n % 1000 == 0:
            print(f"  {split_key}: {n}/{len(imgs)} imágenes procesadas")

    stats["image_size_top"] = [(f"{a}x{b}", int(c)) for (a, b), c in sizes.most_common(5)]
    stats["samples"] = samples

    area_stats = {}
    for c in range(nc):
        arr = np.asarray(box_areas[c], dtype=float)
        area_stats[c] = {
            "median_area_px": float(np.median(arr)) if arr.size else float("nan"),
            "p10_area_px": float(np.percentile(arr, 10)) if arr.size else float("nan"),
            "p90_area_px": float(np.percentile(arr, 90)) if arr.size else float("nan"),
        }

    return {
        "stats": stats,
        "instances": instances,
        "image_presence": image_presence,
        "class_sets": class_sets,
        "per_image_boxes": per_image_boxes,
        "area_stats": area_stats,
    }


split_results = {}
for split_key in ("train", "val", "test"):
    if split_key in SPLITS:
        print(f"\nEscaneando {split_key} ...")
        img_dir, lbl_dir = SPLITS[split_key]
        split_results[split_key] = scan_split(split_key, img_dir, lbl_dir)

# --- Tabla de instancias por clase ------------------------------------------
counts = pd.DataFrame({
    "class_id": list(range(len(EXPECTED_NAMES))),
    "class_name": EXPECTED_NAMES,
})
for split_key, result in split_results.items():
    counts[f"{split_key}_instances"] = result["instances"]
    counts[f"{split_key}_images"] = result["image_presence"]

counts["total_instances"] = counts[[c for c in counts.columns if c.endswith("_instances")]].sum(axis=1)
max_instances = max(int(counts["total_instances"].max()), 1)
counts["imbalance_vs_max"] = max_instances / counts["total_instances"].replace(0, np.nan)

if "train" in split_results:
    counts["train_median_box_area_px"] = [
        split_results["train"]["area_stats"][c]["median_area_px"] for c in range(len(EXPECTED_NAMES))
    ]

counts.to_csv(REPORTS / "class_counts.csv", index=False)
print("\n=== Instancias por clase ===")
display(counts)

# --- QA por split ------------------------------------------------------------
qa_rows = []
for split_key, result in split_results.items():
    s = result["stats"]
    qa_rows.append({
        "split": split_key,
        "images": s["images"],
        "label_files": s["label_files"],
        "boxes": s["boxes"],
        "missing_labels": s["missing_labels"],
        "orphan_labels": s["orphan_labels"],
        "empty_labels": s["empty_labels"],
        "malformed_lines": s["malformed_lines"],
        "bad_class_ids": s["bad_class_ids"],
        "bad_coords": s["bad_coords"],
        "tiny_boxes": s["tiny_boxes"],
        "images_without_boxes": s["images_without_boxes"],
        "corrupt_images": s["corrupt_images"],
        "non_640": s["non_640"],
        "size_top": s["image_size_top"],
    })
qa = pd.DataFrame(qa_rows)
qa.to_csv(REPORTS / "label_qa.csv", index=False)
print("\n=== QA de labels ===")
display(qa)

with open(REPORTS / "label_qa.json", "w", encoding="utf-8") as f:
    json.dump(
        json_safe({k: v["stats"] for k, v in split_results.items()}),
        f, indent=2, ensure_ascii=False, default=str,
    )

# --- Gráficos ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
order = np.argsort(counts["total_instances"].values)
ax.barh(np.asarray(EXPECTED_NAMES)[order], counts["total_instances"].values[order], color="#2b6cb0")
ax.set_xscale("log")
ax.set_xlabel("Instancias (log)")
ax.set_title("ThermoSolar-PV: instancias por clase")
for i, v in enumerate(counts["total_instances"].values[order]):
    ax.text(v, i, f" {int(v)}", va="center", fontsize=9)
fig.tight_layout()
fig.savefig(REPORTS / "class_distribution.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
all_boxes = np.concatenate([np.asarray(v["per_image_boxes"]) for v in split_results.values()]) if split_results else np.array([0])
if all_boxes.size == 0:
    all_boxes = np.array([0])
max_b = int(max(all_boxes.max(), 1))
ax.hist(all_boxes, bins=np.arange(0, max_b + 2) - 0.5, color="#2f855a", edgecolor="white")
ax.set_xlabel("Cajas por imagen")
ax.set_ylabel("Imágenes")
ax.set_title("Distribución de cajas por imagen (todos los splits)")
fig.tight_layout()
fig.savefig(REPORTS / "boxes_per_image.png", dpi=160)
plt.show()

# --- Co-ocurrencia Single/Multi --------------------------------------------
if "train" in split_results:
    sets_train = split_results["train"]["class_sets"]
    rows = []
    def class_id(*tokens):
        for cid, cname in enumerate(EXPECTED_NAMES):
            c = canon_name(cname)
            if all(t in c for t in tokens):
                return cid
        return None

    pairs = []
    for family in ("hotspot", "diode", "bypass"):
        a = class_id("single", family)
        b = class_id("multi", family)
        if a is not None and b is not None:
            pairs.append((a, b))

    for a, b in pairs:
        both = sum(1 for s in sets_train if a in s and b in s)
        only_a = sum(1 for s in sets_train if a in s and b not in s)
        only_b = sum(1 for s in sets_train if b in s and a not in s)
        rows.append({
            "pair": f"{EXPECTED_NAMES[a]} vs {EXPECTED_NAMES[b]}",
            "both_in_same_image": both,
            "only_first": only_a,
            "only_second": only_b,
            "total_first": only_a + both,
            "total_second": only_b + both,
        })
    cooc = pd.DataFrame(rows)
    cooc.to_csv(REPORTS / "single_multi_cooccurrence.csv", index=False)
    print("\n=== Co-ocurrencia Single/Multi en train ===")
    display(cooc)

    n_multi_class = sum(1 for s in sets_train if len(s) > 1)
    print(f"Imágenes de train con más de una clase: {n_multi_class}/{len(sets_train)} "
          f"({100 * n_multi_class / max(len(sets_train), 1):.1f}%)")

print("\nReportes QA guardados en:", REPORTS)


## 5. Auditoría de fuga de información (duplicados y near-duplicates)

El dataset de Kaggle declara ~2,723 imágenes originales y ~7,500 muestras aumentadas. Si los aumentos se generaron **antes** de partir train/valid/test, hay imágenes casi idénticas en splits distintos y las métricas quedan infladas. Esto podría explicar parte de la brecha entre mAP50 y mAP50-95 y la matriz de confusión sospechosa.

El chequeo usa **pHash** (robusto a brillo/escala) + **MD5** (duplicado exacto). Cuenta grupos que cruzan splits y guarda un CSV y ejemplos visuales para que puedas verificar si son realmente el mismo panel.

Esto es un **diagnóstico**, no cambia el entrenamiento salvo que actives `REBUILD_GROUP_SAFE_SPLIT=True`.

> **Limitación:** MD5 detecta copias exactas y pHash detecta cambios de brillo/escala, pero un flip o una rotación de 90°/180° puede cambiar el pHash lo suficiente como para no detectarse. Por eso el porcentaje es una **cota inferior** de la fuga, no un máximo. Aun así, los aumentos guardados como archivos nuevos suelen coincidir por MD5 o pHash.


In [ ]:
def phash64(path: Path, hash_size: int = 8, factor: int = 4):
    """pHash de 64 bits con OpenCV (DCT). Devuelve None si la imagen no se puede leer."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (hash_size * factor, hash_size * factor), interpolation=cv2.INTER_AREA)
    dct = cv2.dct(np.float32(img))
    low = dct[:hash_size, :hash_size]
    med = np.median(low.flatten()[1:])
    bits = (low > med).flatten().astype(np.uint8)
    return int.from_bytes(np.packbits(bits).tobytes(), "big")


_POPCOUNT = np.array([int(i).bit_count() for i in range(256)], dtype=np.uint8)


def popcount64(x: np.ndarray):
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    return _POPCOUNT[x.view(np.uint8).reshape(*x.shape, 8)].sum(axis=-1)


def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


records = []
for split_key in ("train", "val", "test"):
    if split_key not in SPLITS:
        continue
    for img in list_images(SPLITS[split_key][0]):
        records.append({"split": split_key, "path": str(img)})

print(f"Calculando hashes de {len(records)} imágenes ...")
valid_records = []
for i, rec in enumerate(records, 1):
    p = Path(rec["path"])
    h = phash64(p)
    if h is None:
        print(f"  No se pudo leer: {p}")
        continue
    rec = dict(rec)
    rec["phash"] = h
    rec["md5"] = md5_file(p)
    valid_records.append(rec)
    if i % 1000 == 0:
        print(f"  {i}/{len(records)}")

records = valid_records
df_hash = pd.DataFrame(records)
n = len(df_hash)
print(f"Imágenes hasheadas: {n}")

if n == 0:
    raise RuntimeError("No se pudieron leer imágenes para la auditoría de fuga.")

splits_arr = df_hash["split"].to_numpy()
hashes = df_hash["phash"].to_numpy(dtype=np.uint64)

# Union-Find
parent = list(range(n))
rank = [0] * n


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    if rank[ra] < rank[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    if rank[ra] == rank[rb]:
        rank[ra] += 1


# Duplicados exactos
first_by_md5 = {}
exact_cross = 0
for i, h in enumerate(df_hash["md5"].tolist()):
    if h in first_by_md5:
        j = first_by_md5[h]
        union(i, j)
        if splits_arr[i] != splits_arr[j]:
            exact_cross += 1
    else:
        first_by_md5[h] = i

# Near-duplicates entre splits
cross_pairs = 0
for i0 in range(0, n, 256):
    block = hashes[i0:i0 + 256, None] ^ hashes[None, :]
    dist = popcount64(block)
    ii, jj = np.nonzero(dist <= LEAK_HAMMING)
    a_arr = ii + i0
    keep = (a_arr < jj) & (splits_arr[a_arr] != splits_arr[jj])
    for a, b in zip(a_arr[keep], jj[keep]):
        union(int(a), int(b))
        cross_pairs += 1
    if (i0 // 256 + 1) % 4 == 0:
        print(f"  Pares evaluados: {min(i0 + 256, n)}/{n} | cruces acumulados={cross_pairs}")

groups = defaultdict(list)
for i in range(n):
    groups[find(i)].append(i)

leak_groups = []
for g in groups.values():
    if len({splits_arr[i] for i in g}) > 1:
        leak_groups.append(g)
leak_groups.sort(key=len, reverse=True)

n_val = int((splits_arr == "val").sum())
leaked_val = 0
for g in leak_groups:
    gs = {splits_arr[i] for i in g}
    if "val" in gs and "train" in gs:
        leaked_val += sum(1 for i in g if splits_arr[i] == "val")

print("\n=== Auditoría de fuga ===")
print(f"Duplicados exactos que cruzan splits: {exact_cross}")
print(f"Pares near-duplicate que cruzan splits (hamming <= {LEAK_HAMMING}): {cross_pairs}")
print(f"Grupos que cruzan splits: {len(leak_groups)}")
print(f"Imágenes de val con near-duplicate en train: {leaked_val}/{n_val} "
      f"({100 * leaked_val / max(n_val, 1):.1f}%)")

leak_rows = []
for gi, g in enumerate(leak_groups):
    split_counts = Counter(splits_arr[i] for i in g)
    leak_rows.append({
        "group_id": gi,
        "n_images": len(g),
        "n_train": split_counts.get("train", 0),
        "n_val": split_counts.get("val", 0),
        "n_test": split_counts.get("test", 0),
        "splits": ",".join(sorted(split_counts)),
        "sample_files": " | ".join(df_hash.iloc[i]["path"] for i in g[:5]),
    })
leak_df = pd.DataFrame(leak_rows)
leak_df.to_csv(REPORTS / "leakage_groups.csv", index=False)
if len(leak_df):
    print("\nGrupos más grandes:")
    display(leak_df.head(10))

if LEAK_EXAMPLES and leak_groups:
    top = leak_groups[:3]
    ncols = 4
    fig, axes = plt.subplots(len(top), ncols, figsize=(12, 3 * len(top)))
    axes = np.atleast_2d(axes)
    for r, g in enumerate(top):
        for c in range(ncols):
            ax = axes[r, c]
            ax.axis("off")
            if c >= len(g):
                continue
            i = g[c]
            img = cv2.imread(df_hash.iloc[i]["path"], cv2.IMREAD_GRAYSCALE)
            if img is not None:
                ax.imshow(img, cmap="gray")
                ax.set_title(f"{splits_arr[i]} | grupo {r}", fontsize=9)
    fig.suptitle("Ejemplos de grupos near-duplicate que cruzan splits", fontsize=12)
    fig.tight_layout()
    fig.savefig(REPORTS / "leakage_examples.png", dpi=150)
    plt.show()

with open(REPORTS / "leakage_summary.json", "w", encoding="utf-8") as f:
    json.dump(json_safe({
        "n_images": n,
        "exact_cross_split_pairs": exact_cross,
        "near_duplicate_cross_split_pairs": cross_pairs,
        "cross_split_groups": len(leak_groups),
        "val_images_with_train_near_duplicate": leaked_val,
        "val_images_total": n_val,
        "leak_hamming": LEAK_HAMMING,
    }), f, indent=2, ensure_ascii=False)


## 6. Split group-safe (opcional, solo si hay fuga)

Si la celda anterior muestra una fuga alta (>5–10% de val con near-duplicate en train), las métricas del split original son optimistas. Para medir generalización real, activa `REBUILD_GROUP_SAFE_SPLIT=True` y vuelve a ejecutar desde la configuración.

Esto reconstruye `train/valid/test` manteniendo cada grupo de near-duplicates **completo** en un solo split, con proporciones 70/20/10. No es comparable con el 58.02% histórico porque cambia el split; sirve para saber cuánto del resultado era fuga.


In [ ]:
TRAIN_DATA_YAML = DATA_YAML
GROUP_SAFE_YAML = None

if REBUILD_GROUP_SAFE_SPLIT:
    if "df_hash" not in globals() or len(df_hash) == 0:
        raise RuntimeError("Ejecuta primero la auditoría de fuga.")

    group_of = np.array([find(i) for i in range(n)])
    unique_groups = sorted(set(group_of.tolist()))
    rng = random.Random(SEED)
    rng.shuffle(unique_groups)

    target = np.array([0.70, 0.20, 0.10])
    assigned_counts = np.zeros(3, dtype=np.float64)
    group_split = {}

    for gid in unique_groups:
        size = int((group_of == gid).sum())
        scores = (assigned_counts + size) / target
        s = int(np.argmin(scores))
        group_split[gid] = ("train", "val", "test")[s]
        assigned_counts[s] += size

    group_root = WORK / "dataset_group_safe"
    for split_key in ("train", "val", "test"):
        (group_root / split_key / "images").mkdir(parents=True, exist_ok=True)
        (group_root / split_key / "labels").mkdir(parents=True, exist_ok=True)

    copied = Counter()
    for i, rec in enumerate(df_hash.itertuples(index=False)):
        split_key = group_split[group_of[i]]
        src_img = Path(rec.path)
        src_lbl = SPLITS[rec.split][1] / f"{src_img.stem}.txt"
        dst_img = group_root / split_key / "images" / f"{i:06d}_{src_img.name}"
        dst_lbl = group_root / split_key / "labels" / f"{i:06d}_{src_img.stem}.txt"
        shutil.copy2(src_img, dst_img)
        if src_lbl.exists():
            shutil.copy2(src_lbl, dst_lbl)
        copied[split_key] += 1

    group_yaml = {
        "path": str(group_root),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "names": {i: n for i, n in enumerate(EXPECTED_NAMES)},
    }
    GROUP_SAFE_YAML = WORK / "data_group_safe.yaml"
    with open(GROUP_SAFE_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(group_yaml, f, sort_keys=False, allow_unicode=True)

    TRAIN_DATA_YAML = GROUP_SAFE_YAML
    print("Split group-safe creado:", GROUP_SAFE_YAML)
    print("Imágenes por split:", dict(copied))
    print("ADVERTENCIA: las métricas de este split NO son comparables con el 58.02% histórico.")
else:
    print("Split original conservado. Comparabilidad con el baseline: SÍ.")
    print("TRAIN_DATA_YAML =", TRAIN_DATA_YAML)


## 7. Baseline: `best.pt` actual a 320 vs 640 (opcional)

Si subiste el `best.pt` actual como Kaggle Dataset, esta celda comprueba la hipótesis principal del handoff: el modelo se entrenó a 320 y se usa a 640. Se evalúa el **mismo checkpoint** a ambas resoluciones, con `conf=0.001` (protocolo de validación de Ultralytics, no 0.25) y se guardan las métricas por clase.

Si no hay `best.pt` adjunto, la celda se salta sin romper el notebook.


In [ ]:
baseline_weights = find_attached_pt(BASELINE_WEIGHTS) if RUN_BASELINE else None
baseline_rows = []
baseline_tables = {}

if baseline_weights is None:
    print("No hay best.pt adjunto; se salta el baseline 320 vs 640.")
else:
    print("Baseline:", baseline_weights)
    tmp = YOLO(str(baseline_weights))
    ckpt_names = list(tmp.names.values()) if isinstance(tmp.names, dict) else list(tmp.names)
    print("Clases del checkpoint:", ckpt_names)
    if [canon_name(x) for x in ckpt_names] != [canon_name(x) for x in EXPECTED_NAMES]:
        warnings.warn("El best.pt adjunto no tiene el orden canónico. Revisa antes de comparar.")

    for sz in (320, 640):
        print(f"\nValidando baseline a {sz} px ...")
        model_b = YOLO(str(baseline_weights))
        m = model_b.val(
            data=str(DATA_YAML),
            imgsz=sz,
            batch=BATCH,
            conf=0.001,
            iou=0.7,
            plots=True,
            project=str(WORK / "runs_baseline"),
            name=f"val_{sz}",
            exist_ok=True,
            verbose=True,
        )
        table = per_class_table(m)
        table.to_csv(REPORTS / f"baseline_{sz}_per_class.csv", index=False)
        baseline_tables[sz] = table
        baseline_rows.append({
            "imgsz": sz,
            "precision": float(m.box.mp),
            "recall": float(m.box.mr),
            "mAP50": float(m.box.map50),
            "mAP75": float(m.box.map75),
            "mAP50-95": float(m.box.map),
        })
        print(f"  {sz}px: mAP50={m.box.map50:.4f} | mAP50-95={m.box.map:.4f} | P={m.box.mp:.4f} | R={m.box.mr:.4f}")

    baseline_df = pd.DataFrame(baseline_rows)
    baseline_df.to_csv(REPORTS / "baseline_320_vs_640.csv", index=False)
    display(baseline_df)

    for sz, table in baseline_tables.items():
        print(f"\nPor clase a {sz} px:")
        display(table)

# Liberar VRAM del baseline antes de entrenar (la T4 es de 16 GB).
try:
    del tmp, model_b
except NameError:
    pass
torch.cuda.empty_cache()


## 8. Entrenamiento YOLOv9-S a 640

Esta es la celda principal. Si existe `runs/<RUN_NAME>/weights/last.pt`, **reanuda** automáticamente. Si no, empieza desde `yolov9s.pt` (o desde `INIT_WEIGHTS` si lo cambias).

- `TIME_LIMIT_H=8.5` evita que Kaggle mate la sesión a mitad de época. Si se alcanza el límite, `last.pt` queda guardado y la próxima ejecución continúa.
- `save_period=10` guarda checkpoints numerados; `last.pt` se actualiza cada época.
- Se hace una copia de seguridad en `/kaggle/working/checkpoints/last.pt` cada 10 épocas.


In [ ]:
RUN_DIR = PROJECT / RUN_NAME
BACKUP_LAST = BACKUP_DIR / "last.pt"
last_local = RUN_DIR / "weights" / "last.pt"

# Restaurar desde backup o desde un last.pt adjunto.
# OJO: no creamos RUN_DIR aquí; si existe vacío, Ultralytics lo incrementaría a "-2".
if not last_local.exists() and BACKUP_LAST.exists():
    (RUN_DIR / "weights").mkdir(parents=True, exist_ok=True)
    shutil.copy2(BACKUP_LAST, last_local)
    print("Restaurado last.pt desde backup:", last_local)

if not last_local.exists() and RESUME_WEIGHTS:
    src = Path(RESUME_WEIGHTS)
    if not src.exists():
        raise FileNotFoundError(f"RESUME_WEIGHTS no existe: {src}")
    (RUN_DIR / "weights").mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, last_local)
    print("Copiado last.pt desde RESUME_WEIGHTS:", last_local)


def backup_last(trainer):
    try:
        if SAVE_PERIOD and (trainer.epoch + 1) % SAVE_PERIOD == 0:
            src = Path(trainer.save_dir) / "weights" / "last.pt"
            if src.exists():
                shutil.copy2(src, BACKUP_LAST)
    except Exception as e:
        print("Aviso: no se pudo copiar el backup de last.pt:", e)


def load_ckpt(path: Path):
    try:
        try:
            return torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            return torch.load(path, map_location="cpu")
    except Exception as e:
        print("No se pudo leer el checkpoint:", e)
        return {}


def ckpt_epoch(path: Path):
    return int(load_ckpt(path).get("epoch", -1))


def ckpt_class_names(path: Path):
    """Nombres de clase guardados en el checkpoint, o None si no se pueden leer."""
    ckpt = load_ckpt(path)
    names = None
    if isinstance(ckpt, dict):
        train_args = ckpt.get("train_args")
        if isinstance(train_args, dict):
            names = train_args.get("names")
        if names is None:
            names = ckpt.get("names")
        if names is None:
            inner = ckpt.get("ema") or ckpt.get("model")
            names = getattr(inner, "names", None)
    if isinstance(names, dict):
        try:
            return [str(names[k]) for k in sorted(names, key=lambda x: int(x))]
        except Exception:
            return [str(v) for v in names.values()]
    if isinstance(names, (list, tuple)):
        return [str(x) for x in names]
    return None


_TRAIN_T0 = {"t": None}


def prepare_resume_checkpoint(path: Path, total_epochs: int):
    """Ajusta train_args del checkpoint para continuar el schedule hasta total_epochs.

    Un run cortado por `time` guarda args.epochs = las épocas que cupieron y
    args.time > 0. Ultralytics, al reanudar, restaura esos valores y recalcula
    self.epochs desde `time` en cada época, comparándolo con la época ABSOLUTA:
    el entrenamiento se detendría tras 1 época (o fallaría el assert de resume).
    Por eso forzamos epochs=total_epochs y time=0.
    """
    ckpt = load_ckpt(path)
    train_args = ckpt.get("train_args") if isinstance(ckpt, dict) else None
    if not isinstance(train_args, dict):
        print("Aviso: no se pudo leer train_args; se reanuda sin ajustar.")
        return
    old_epochs, old_time = train_args.get("epochs"), train_args.get("time")
    train_args["epochs"] = int(total_epochs)
    train_args["time"] = 0
    if old_epochs == train_args["epochs"] and old_time == train_args["time"]:
        return
    backup = path.with_name(path.stem + "_orig.pt")
    if not backup.exists():
        shutil.copy2(path, backup)
        print("Backup del checkpoint original:", backup)
    torch.save(ckpt, path)
    print(
        f"Checkpoint ajustado: epochs {old_epochs} -> {total_epochs}, time {old_time} -> 0. "
        "La cosine se recalcula sobre el total; puede haber un pequeño salto de LR."
    )


def stop_after_time_limit(trainer):
    """Parada limpia por reloj cuando Ultralytics no puede usar args.time (resume)."""
    if not TIME_LIMIT_H or TIME_LIMIT_H <= 0:
        return
    if _TRAIN_T0["t"] is None:
        _TRAIN_T0["t"] = time.time()
    elapsed_h = (time.time() - _TRAIN_T0["t"]) / 3600
    if elapsed_h >= TIME_LIMIT_H:
        trainer.stop = True
        print(f"TIME_LIMIT_H alcanzado ({elapsed_h:.2f} h); parada limpia al final de la época.")


# Un last.pt del orden antiguo no se puede reanudar con el data.yaml actual:
# `resume=True` restaura el orden de clases del checkpoint y desalinearía los IDs.
if last_local.exists():
    ckpt_names = ckpt_class_names(last_local)
    if ckpt_names and [canon_name(x) for x in ckpt_names] != [canon_name(x) for x in EXPECTED_NAMES]:
        raise ValueError(
            "El last.pt existente usa otro orden de clases que el data.yaml actual.\n"
            f"  checkpoint: {ckpt_names}\n"
            f"  activo    : {EXPECTED_NAMES}\n"
            "Borra o renombra ese run, o cambia RUN_NAME, antes de entrenar. "
            "No reanudes con un orden distinto."
        )


start = time.time()
results = None
model = None

if AUTO_RESUME and last_local.exists() and ckpt_epoch(last_local) + 1 >= EPOCHS:
    print(f"El run ya alcanzó {EPOCHS} épocas en {last_local}; se omite el entrenamiento.")
elif AUTO_RESUME and last_local.exists():
    start_epoch = ckpt_epoch(last_local) + 1
    prepare_resume_checkpoint(last_local, RESUME_TOTAL_EPOCHS)
    print(f"Reanudando desde {last_local} (época {start_epoch} de {RESUME_TOTAL_EPOCHS}) ...")
    _TRAIN_T0["t"] = time.time()
    model = YOLO(str(last_local))
    model.add_callback("on_train_epoch_end", backup_last)
    if TIME_LIMIT_H and TIME_LIMIT_H > 0:
        model.add_callback("on_train_epoch_end", stop_after_time_limit)
    results = model.train(
        resume=True,
        time=0,  # con 'time' Ultralytics recalcula epochs y corta al reanudar
        close_mosaic=RESUME_TOTAL_EPOCHS if RESUME_KEEP_MOSAIC_CLOSED else CLOSE_MOSAIC,
        data=str(TRAIN_DATA_YAML),
    )
else:
    # Evitar pisar un run existente sin checkpoint y evitar que un directorio vacío
    # haga que Ultralytics guarde en "RUN_NAME-2" en lugar de "RUN_NAME".
    if RUN_DIR.exists():
        if any(RUN_DIR.iterdir()):
            RUN_DIR = PROJECT / f"{RUN_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
            print("Run existente sin last.pt; se usa uno nuevo:", RUN_DIR)
        else:
            RUN_DIR.rmdir()
            print("Directorio vacío eliminado; se reutiliza el nombre:", RUN_NAME)

    init_path = str(INIT_WEIGHTS)
    if init_path == "best.pt":
        found = find_attached_pt(BASELINE_WEIGHTS)
        if found is None:
            raise FileNotFoundError("INIT_WEIGHTS='best.pt' pero no hay best.pt adjunto.")
        init_path = str(found)

    print(f"Iniciando entrenamiento desde {init_path} en {RUN_DIR} ...")
    model = YOLO(init_path)
    model.add_callback("on_train_epoch_end", backup_last)

    train_kwargs = dict(
        data=str(TRAIN_DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=(DEVICE_LIST if len(DEVICE_LIST) > 1 else DEVICE_LIST[0]),
        workers=WORKERS,
        amp=True,
        cache=False,
        patience=PATIENCE,
        cos_lr=True,
        close_mosaic=CLOSE_MOSAIC,
        save=True,
        save_period=SAVE_PERIOD,
        val=True,
        plots=True,
        project=str(PROJECT),
        name=RUN_DIR.name,
        exist_ok=False,
        seed=SEED,
        deterministic=False,
        optimizer="auto",
        lr0=LR0,
        cls_pw=CLS_PW,
        nbs=64,
        rect=False,
        single_cls=False,
        fraction=1.0,
        pretrained=True,
        verbose=True,
        **AUG,
    )
    if TIME_LIMIT_H and TIME_LIMIT_H > 0:
        train_kwargs["time"] = TIME_LIMIT_H

    results = model.train(**train_kwargs)

if model is not None and getattr(model, "trainer", None) is not None:
    RUN_DIR = Path(model.trainer.save_dir)

elapsed_h = (time.time() - start) / 3600
print(f"\nEntrenamiento terminado en {elapsed_h:.2f} h")
print("RUN_DIR:", RUN_DIR)
print("best.pt:", (RUN_DIR / "weights" / "best.pt").exists())
print("last.pt:", (RUN_DIR / "weights" / "last.pt").exists())

if results is not None and hasattr(results, "results_dict"):
    with open(REPORTS / "train_results.json", "w", encoding="utf-8") as f:
        json.dump(json_safe(results.results_dict), f, indent=2, ensure_ascii=False, default=str)
    print("Resultados de train:", results.results_dict)

if TIME_LIMIT_H and elapsed_h >= TIME_LIMIT_H * 0.98:
    print("\nSe alcanzó TIME_LIMIT_H. Vuelve a ejecutar esta celda en la misma sesión o en una nueva:")
    print("AUTO_RESUME=True reanudará desde last.pt. Si /kaggle/working se reinició, sube last.pt")
    print("como Kaggle Dataset y define RESUME_WEIGHTS con su ruta.")


## 9. Evaluación final, métricas por clase y matriz de confusión

Se evalúa `best.pt` con el protocolo estándar de Ultralytics (`conf=0.001`, `iou=0.7`). La matriz de confusión se guarda **con etiquetas** y se exporta a CSV. Recuerda la orientación de Ultralytics: **filas = predicho, columnas = real**, y la última fila/columna es `background`. Esto es justo lo que hay que verificar para resolver la alerta del handoff.


In [ ]:
BEST = RUN_DIR / "weights" / "best.pt"
if not BEST.exists():
    # Fallback: si el run se guardó con otro sufijo (p. ej. RUN_NAME-2), busca el más reciente.
    candidates = sorted(
        PROJECT.glob(f"{RUN_NAME}*/weights/best.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        BEST = candidates[0]
        RUN_DIR = BEST.parent.parent
        print("best.pt no estaba en RUN_DIR; usando el más reciente:", BEST)

if not BEST.exists():
    raise FileNotFoundError(f"No existe {BEST}. Revisa el entrenamiento y RUN_DIR={RUN_DIR}.")

# Libera el modelo/entrenador anterior antes de cargar best.pt para la evaluación.
try:
    del model
except NameError:
    pass
torch.cuda.empty_cache()

model = YOLO(str(BEST))
m = model.val(
    data=str(TRAIN_DATA_YAML),
    imgsz=IMG_SIZE,
    batch=BATCH,
    conf=0.001,
    iou=0.7,
    plots=True,
    project=str(WORK / "runs_eval"),
    name="final_val",
    exist_ok=True,
    verbose=True,
)
EVAL_DIR = Path(m.save_dir)
print("EVAL_DIR:", EVAL_DIR)

final_table = per_class_table(m)
final_table.to_csv(REPORTS / "final_per_class.csv", index=False)

overall = {
    "precision": float(m.box.mp),
    "recall": float(m.box.mr),
    "mAP50": float(m.box.map50),
    "mAP75": float(m.box.map75),
    "mAP50-95": float(m.box.map),
    "speed_ms": {k: float(v) for k, v in m.speed.items()},
}
print("\n=== Métricas globales ===")
print(json.dumps(overall, indent=2))
display(final_table)

# --- Matriz de confusión con etiquetas -------------------------------------
cm = getattr(m, "confusion_matrix", None)
if cm is not None:
    cm.plot(normalize=True, save_dir=str(EVAL_DIR))
    cm.plot(normalize=False, save_dir=str(EVAL_DIR))

    cm_counts = pd.DataFrame(cm.summary(normalize=False, decimals=0))
    cm_norm = pd.DataFrame(cm.summary(normalize=True, decimals=4))
    cm_counts.to_csv(REPORTS / "confusion_matrix_counts.csv", index=False)
    cm_norm.to_csv(REPORTS / "confusion_matrix_normalized.csv", index=False)

    labels = [m.names[i] for i in range(len(m.names))] + ["background"]
    mat = np.rint(np.asarray(cm.matrix)).astype(int)
    mat_df = pd.DataFrame(mat, index=[f"pred:{x}" for x in labels], columns=[f"true:{x}" for x in labels])
    print("\n=== Matriz de confusión (filas=predicho, columnas=real) ===")
    display(mat_df)

    print("\n=== Matriz normalizada (por columna real) ===")
    display(cm_norm)
else:
    print("Aviso: esta versión de Ultralytics no expuso confusion_matrix; revisa las imágenes en", EVAL_DIR)

# --- Comparación contra el run 320 histórico -------------------------------
REFERENCE_320 = {
    "mAP50": 0.7610,
    "mAP50-95": 0.5802,
    "precision": 0.7674,
    "recall": 0.7188,
}
REFERENCE_PER_CLASS_CHECKPOINT = {
    0: (0.9337, 0.7523),
    1: (0.4455, 0.3514),
    2: (0.7197, 0.5720),
    3: (0.9019, 0.7130),
    4: (0.7672, 0.5959),
    5: (0.8104, 0.6251),
    6: (0.8052, 0.5927),
    7: (0.7057, 0.4246),
}
# La referencia histórica está indexada por ID de label. Si el split es el mismo,
# los IDs son los mismos y la comparación correcta es POR ID, no por nombre: los
# nombres del best.pt antiguo pueden estar permutados respecto al data.yaml.
REFERENCE_PER_CLASS_BY_NAME = {
    canon_name(CHECKPOINT_NAMES[i]): v
    for i, v in REFERENCE_PER_CLASS_CHECKPOINT.items()
}

comparison = final_table.copy()
comparison["checkpoint_name"] = comparison["class_id"].map(
    lambda c: CHECKPOINT_NAMES[int(c)] if 0 <= int(c) < len(CHECKPOINT_NAMES) else "?"
)
comparison["ref_mAP50_320"] = comparison["class_id"].map(
    lambda c: REFERENCE_PER_CLASS_CHECKPOINT.get(int(c), (np.nan, np.nan))[0]
)
comparison["ref_mAP50-95_320"] = comparison["class_id"].map(
    lambda c: REFERENCE_PER_CLASS_CHECKPOINT.get(int(c), (np.nan, np.nan))[1]
)
comparison["delta_mAP50"] = comparison["mAP50"] - comparison["ref_mAP50_320"]
comparison["delta_mAP50-95"] = comparison["mAP50-95"] - comparison["ref_mAP50-95_320"]
comparison.to_csv(REPORTS / "comparison_vs_320.csv", index=False)

print(
    "\nNota: la referencia por clase se compara POR ID (mismos labels), no por nombre. "
    "Si los nombres del checkpoint antiguo estaban permutados, 'checkpoint_name' lo evidencia."
)
if not globals().get("OLD_BEST_COMPATIBLE", True):
    print(
        "El data.yaml usa otro ORDEN de nombres que el best.pt antiguo; el baseline 320 vs 640 "
        "no se ejecutó. El mAP agregado solo es comparable si el split de validación es el mismo."
    )

print("\n=== Comparación contra el run 320 (referencia del handoff) ===")
display(comparison)
print("\nGlobal vs referencia 320:")
for k, v in REFERENCE_320.items():
    print(f"  {k}: {overall[k]:.4f} vs {v:.4f}  (delta {overall[k] - v:+.4f})")

if TRAIN_DATA_YAML != DATA_YAML:
    print("\nADVERTENCIA: el split es group-safe; la comparación con el 58.02% histórico NO es válida.")

weak = final_table.sort_values("mAP50-95").head(3)
print("\nClases más débiles (mAP50-95):")
display(weak)

# Filas/columnas de la matriz para las dos clases objetivo.
# cm.summary() sanea los nombres (espacios -> "_"), así que usamos la misma regla.
def cm_key(name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]", "_", name)


def id_for(*tokens):
    for cid, cname in enumerate(EXPECTED_NAMES):
        c = canon_name(cname)
        if all(t in c for t in tokens):
            return cid
    return None


FOCUS_IDS = [i for i in (id_for("multi", "hotspot"), id_for("string", "reversed")) if i is not None]

if cm is not None:
    for cid in FOCUS_IDS:
        name = EXPECTED_NAMES[cid]
        key = cm_key(name)
        print(f"\nClase {cid} ({name}) — matriz normalizada, fila predicha y columna real:")
        if key in cm_norm["Predicted"].values:
            display(cm_norm[cm_norm["Predicted"] == key])
        else:
            print(f"  Aviso: no se encontró la fila '{key}' en cm.summary().")
        if key in cm_norm.columns:
            display(cm_norm[["Predicted", key]])
        else:
            print(f"  Aviso: no se encontró la columna '{key}' en cm.summary().")

with open(REPORTS / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(json_safe({
        "overall": overall,
        "per_class": final_table.to_dict(orient="records"),
        "train_data_yaml": str(TRAIN_DATA_YAML),
        "run_dir": str(RUN_DIR),
        "img_size": IMG_SIZE,
        "epochs": EPOCHS,
        "batch": BATCH,
        "class_order_policy": CLASS_ORDER_POLICY,
        "class_names_active": EXPECTED_NAMES,
        "class_names_checkpoint": CHECKPOINT_NAMES,
        "old_best_compatible": bool(globals().get("OLD_BEST_COMPATIBLE", True)),
    }), f, indent=2, ensure_ascii=False, default=str)


## 10. Evaluación en test y TTA (opcionales)

- **Test**: solo si el dataset tiene `test/labels`. Es la métrica más honesta si el test no se usó para nada.
- **TTA**: `augment=True` en validación. Cuesta ~3× más y no siempre mejora; úsalo como experimento separado.


In [ ]:
if RUN_TEST_EVAL and ("test" in SPLITS or TRAIN_DATA_YAML != DATA_YAML):
    print("Evaluando en test ...")
    mt = model.val(
        data=str(TRAIN_DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH,
        conf=0.001,
        iou=0.7,
        plots=True,
        project=str(WORK / "runs_eval"),
        name="final_test",
        exist_ok=True,
        verbose=True,
    )
    test_table = per_class_table(mt)
    test_table.to_csv(REPORTS / "test_per_class.csv", index=False)
    print(f"TEST: mAP50={mt.box.map50:.4f} | mAP50-95={mt.box.map:.4f}")
    display(test_table)
else:
    print("Test omitido (no hay split test con labels o RUN_TEST_EVAL=False).")

if RUN_TTA_EVAL:
    print("Evaluando con TTA ...")
    mtta = model.val(
        data=str(TRAIN_DATA_YAML),
        imgsz=IMG_SIZE,
        batch=BATCH,
        conf=0.001,
        iou=0.7,
        augment=True,
        plots=True,
        project=str(WORK / "runs_eval"),
        name="final_tta",
        exist_ok=True,
        verbose=True,
    )
    tta_table = per_class_table(mtta)
    tta_table.to_csv(REPORTS / "tta_per_class.csv", index=False)
    print(f"TTA: mAP50={mtta.box.map50:.4f} | mAP50-95={mtta.box.map:.4f}")
    display(tta_table)
else:
    print("TTA omitido (RUN_TTA_EVAL=False).")


## 11. Export ONNX (opcional, para Railway CPU)

El API actual usa PyTorch. Exportar a ONNX puede reducir latencia en CPU, pero **no** cambia `main.py` automáticamente. Si lo activas, tendrás que integrar `onnxruntime` y el postproceso NMS en el servicio.


In [ ]:
if EXPORT_ONNX:
    print("Exportando a ONNX ...")
    onnx_path = Path(model.export(
        format="onnx",
        imgsz=IMG_SIZE,
        opset=12,
        simplify=True,
        dynamic=False,
        half=False,
    ))
    dst = ARTIFACTS / f"{RUN_DIR.name}_{onnx_path.name}"
    shutil.copy2(onnx_path, dst)
    print("ONNX:", dst, f"({dst.stat().st_size / 1024**2:.1f} MB)")
else:
    print("Export ONNX omitido (EXPORT_ONNX=False).")


## 12. Artefactos y descarga

Copia `best.pt`, `last.pt`, `results.csv`, plots y reportes a `/kaggle/working/artifacts`, crea un ZIP y muestra enlaces de descarga. En Kaggle también puedes usar la pestaña **Output** del notebook.


In [ ]:
ARTIFACTS.mkdir(parents=True, exist_ok=True)
run_tag = RUN_DIR.name
run_artifacts = ARTIFACTS / run_tag
run_artifacts.mkdir(parents=True, exist_ok=True)
copied_files = []

# Pesos
for wname in ("best.pt", "last.pt"):
    src = RUN_DIR / "weights" / wname
    if src.exists():
        dst = run_artifacts / wname
        shutil.copy2(src, dst)
        copied_files.append(dst)

# Archivos del run
for fname in (
    "results.csv",
    "args.yaml",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "labels.jpg",
    "labels_correlogram.jpg",
    "train_batch0.jpg",
    "val_batch0_pred.jpg",
    "val_batch0_labels.jpg",
):
    src = RUN_DIR / fname
    if src.exists():
        dst = run_artifacts / fname
        shutil.copy2(src, dst)
        copied_files.append(dst)

# YAML de datos realmente usados (reproducibilidad)
for yml in {Path(DATA_YAML), Path(TRAIN_DATA_YAML)}:
    if yml.exists():
        dst = run_artifacts / yml.name
        shutil.copy2(yml, dst)
        copied_files.append(dst)

# Reportes
for src in REPORTS.glob("*"):
    if src.is_file():
        dst = run_artifacts / src.name
        shutil.copy2(src, dst)
        copied_files.append(dst)

# Reportes de evaluación
if "EVAL_DIR" in globals() and Path(EVAL_DIR).exists():
    for src in Path(EVAL_DIR).glob("*"):
        if src.is_file() and src.suffix.lower() in {".png", ".csv", ".json", ".jpg"}:
            dst = run_artifacts / f"eval_{src.name}"
            shutil.copy2(src, dst)
            copied_files.append(dst)

zip_path = WORK / f"{run_tag}_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(run_artifacts.glob("*")):
        if p.is_file():
            zf.write(p, arcname=p.name)

best_artifact = run_artifacts / "best.pt"
print(f"Artefactos: {len(copied_files)} archivos en {ARTIFACTS}")
print(f"ZIP: {zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MB)")
if best_artifact.exists():
    sha = hashlib.sha256(best_artifact.read_bytes()).hexdigest()
    print(f"best.pt SHA256: {sha}")
    print(f"best.pt: {best_artifact} ({best_artifact.stat().st_size / 1024**2:.1f} MB)")

# --- Descarga ---------------------------------------------------------------
# Kaggle NO resuelve FileLink con rutas absolutas: el enlace apunta a
# https://www.kaggleusercontent.com/kaggle/working/... y da 404. Hay que usar
# rutas RELATIVAS a /kaggle/working.
os.chdir(WORK)
print("cwd para descargas:", Path.cwd())

# Copia best.pt y el ZIP a la raíz de /kaggle/working para que aparezcan
# directamente en el panel Output.
for p in (best_artifact, zip_path):
    if p.exists() and p.parent != WORK:
        dst = WORK / p.name
        if not dst.exists() or dst.stat().st_size != p.stat().st_size:
            shutil.copy2(p, dst)


def kaggle_link(p: Path):
    """Muestra un FileLink válido en Kaggle (ruta relativa a /kaggle/working)."""
    p = Path(p)
    if not p.exists():
        print(f"No existe: {p}")
        return
    try:
        rel = p.relative_to(WORK)
    except ValueError:
        print(f"Fuera de /kaggle/working; Kaggle no lo puede servir: {p}")
        return
    print(f"Descargar: {p} ({p.stat().st_size / 1024**2:.1f} MB)")
    display(FileLink(str(rel)))


kaggle_link(WORK / best_artifact.name)
kaggle_link(zip_path)
if (run_artifacts / "last.pt").exists():
    kaggle_link(run_artifacts / "last.pt")

print(
    "\nSi el enlace sigue sin descargar: panel derecho -> Output -> "
    f"{best_artifact.name} o {zip_path.name} -> Download.\n"
    "El ZIP contiene best.pt, last.pt, results.csv, plots y reports."
)


## 13. Criterio de decisión y siguientes pasos

### Cómo leer los resultados

| Señal | Interpretación | Acción |
|---|---|---|
| `baseline_640` > `baseline_320` | Confirma el desajuste train/infer | Mantener inferencia a 640; el reentrenamiento a 640 debería subir |
| Nuevo `mAP50-95` > 58.02% | P0 funcionó | Continuar con P1/P2 |
| ID 1 (`Multi Diode`) sigue < 50% mAP50 | El problema es la frontera Single/Multi o la definición de la clase | P1: revisar confusión ID 1 ↔ ID 4 antes de reformular |
| `String Reversed Polarity` sigue < 50% mAP50-95 | Clase rara / granularidad de string | P2: oversampling + modelo separado string-level |
| Fuga > 5–10% de val | Métricas históricas optimistas | Reentrenar con split group-safe y reportar ambos |
| `precision` alta y `recall` baja | Umbral/confianza o clases raras | Ajustar `conf` en producción y revisar PR curves |

### P1 recomendado (siguiente notebook, no este)

La clase más débil es **ID 1 = `Multi Diode`** (mAP50 0.474, mAP50-95 0.346), no “Multi Hotspots”. Antes de cambiar el modelo:

1. Revisar `confusion_matrix_normalized.csv`: **ID 1 vs ID 4** (`Multi Diode` vs `Single Diode`). Si se confunden, el problema es la frontera Single/Multi, no la capacidad del detector.
2. Revisar las cajas: los diodos son rectángulos muy finos (`height` ≈ 0.02–0.10 en el correlograma), así que la localización a 640 penaliza AP75/AP50-95.
3. Si la confusión es alta, probar el esquema de 5 clases base + clasificador Single/Multi. Si no, mantener 8 clases y atacar datos/etiquetas.
4. Para `OpenCircuit`/`ReversedPolarity`, evaluar un modelo separado a nivel de string.

### P2 recomendado

- Oversampling de imágenes que contengan `Multi Diode`, `String Reversed Polarity` y `String Open Circuit`.
- `cls_pw=0.25–0.5` como experimento B (una variable a la vez).
- Revisar el protocolo de etiquetado: ¿una caja por anomalía o una caja por panel/string?

### Cosas que NO hacer

- No usar Soup para entrenar YOLO ni para mejorar `best.pt`.
- No usar un VLM para devolver bounding boxes.
- No entrenar a 320 e inferir a 640.
- No confiar en la matriz de confusión sin etiquetas: la de este notebook tiene filas=predicho, columnas=real, y `background` al final.
- No hacer el run final a 640 en la RTX 3050 de 4 GB.
- No cambiar de framework antes de agotar resolución, datos y formulación de clases.

### Nota sobre `main.py`

`main.py` ya usa `imgsz=640` y el orden de clases confirmado con `labels.jpg` (`Multi Bypassed Substring`, `Multi Diode`, `Multi Hotspots`, …). Si el nuevo `best.pt` se entrena a 640, no hay que cambiar la resolución de inferencia. Si vuelves a cambiar el orden, actualiza `CLASS_NAMES` y vuelve a validar la matriz de confusión.
